# Sentiment Analysis

Sentiment is a type of classification concerned with determining the polarity of an piece of text, i.e. if it is positive, negative or neutral.

Here we use a model developed by the University of Cardiff's NLP group: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest

In [22]:
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
from scipy.special import softmax
import numpy as np
import pandas as pd
from tqdm import tqdm
import csv
import ast

In [23]:
df=pd.read_csv('phd data/merged post comments.csv', encoding='utf-8')
df.head()

,Unnamed: 0,account,Date,post url,urls,text_x,Likes,Comments,hashtags,mentions,tagged users,is video,video url,caption.created_at_formatted,text_y,comments_text
0,0,adis.a93,2022-03-10 20:34:00,Ca79j-DruOL,https://scontent-fra5-1.cdninstagram.com/v/t51...,Hannover zeigt Menschlichkeit und Hilfsbereits...,985.0,632.0,['ukraine'],"['regionspraesident', 'steffenkrach', 'tomek_l...","['steffenkrach', 'regionspraesident']",False,NaN,2022/03/10 20:34:52,Hannover zeigt Menschlichkeit und Hilfsbereits...,"['Europe is preparing to celebrate Easter, Ukr..."
1,1,adis.a93,2022-03-08 16:05:00,Ca2VIVgLeTJ,https://scontent-fra3-1.cdninstagram.com/v/t51...,Es gibt keinen Erfolg und Frieden ohne Frauen!...,2165.0,189.0,[],[],['tim.wook'],False,NaN,2022/03/08 16:05:22,Es gibt keinen Erfolg und Frieden ohne Frauen!...,"['👏👏👏👏👏', 'In einer Zeit, in der viele Länder ..."
2,2,adis.a93,2022-02-26 13:21:00,CacSefPN61Z,https://scontent-fra3-2.cdninstagram.com/v/t51...,Die Bilder aus Kiew erinnern mich an Sarajevo ...,1239.0,166.0,"['ukraine', 'russia', 'war', 'peace']",['tomek_lip'],"['stephan.weil.spd', 'tim.wook', 'stefanpolitz...",False,NaN,2022/02/26 13:21:55,Die Bilder aus Kiew erinnern mich an Sarajevo ...,['We are Ukrainians and we need your help! rus...
3,3,adis.a93,2022-02-24 18:08:00,CaXprHNr5zA,https://scontent-fra3-1.cdninstagram.com/v/t51...,Ein neues dunkles Kapitel für Europa!\n\nDie r...,1155.0,76.0,"['ukraine', 'russia', 'war', 'peace']",[],[],False,NaN,2022/02/24 18:08:25,Ein neues dunkles Kapitel für Europa!\n\nDie r...,"['In einer Zeit, in der viele Länder russische..."
4,4,adis.a93,2022-02-17 15:03:00,CaFSWMIlyEW,https://scontent-fra3-2.cdninstagram.com/v/t51...,"Wir wollen keinen Krieg, sondern Frieden mit D...",1675.0,160.0,[],[],[],True,https://scontent-fra5-1.cdninstagram.com/o1/v/...,2022/02/17 15:03:46,"Wir wollen keinen Krieg, sondern Frieden mit D...","['👏👏👏', '👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻', 'Unglaublich.... das i..."


In [24]:
model = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model)
config = AutoConfig.from_pretrained(model)

In [25]:
model = AutoModelForSequenceClassification.from_pretrained(model)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [26]:
# 定义情绪分析函数
def get_sentiment_label(text):
    try:
        # 检查文本是否为空
        if not text.strip():
            return None

        # 调用您的情绪分析模型，获取情绪标签
        encoded_input = tokenizer(text, return_tensors='pt')
        # Get model output
        output = model(**encoded_input)
        # Extract scores and apply softmax
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        # Get the sentiment with the highest score
        ranking = np.argsort(scores)[::-1]
        top_label = config.id2label[ranking[0]]
        top_score = scores[ranking[0]]
        return top_label
    except Exception as e:
        # 处理异常
        print(f"Error processing text: {e}")
        return None

In [27]:
# 定义主函数
def analyze_comments(input_file, output_file, start_index=0):
    # 读取输入文件
    df = pd.read_csv(input_file)

    # 如果是从头开始，初始化输出文件并写入表头
    if start_index == 0:
        with open(output_file, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["post url", "comments_list", "sentiment_labels"])  # 写入表头

    # 逐行处理数据
    for index, row in df[start_index:].iterrows():
        print(f"Processing index {index}")

        post_url = row['post url']  # 获取 'post url' 列
        comments_str = row['comments_text']  # 获取 'comments_text' 列

        # 将字符串形式的列表转换为实际的列表
        try:
            comments_list = ast.literal_eval(comments_str)
        except Exception as e:
            print(f"Error parsing comments at index {index}: {e}")
            continue  # 如果解析失败，跳过该行

        # 存储每条评论的情绪标签
        sentiment_labels = []

        # 逐条分析列表中的评论
        for comment in comments_list:
            if comment.strip() == "":  # 检查评论是否为空
                sentiment_labels.append(None)  # 对于空评论，添加 None
                continue  # 跳过空评论

            # 调用情绪分析函数，只返回情绪标签
            sentiment_label = get_sentiment_label(comment)
            sentiment_labels.append(sentiment_label)  # 添加到情绪标签列表中

        # 将结果写入输出文件，包括 post_url、评论列表和情绪标签列表
        with open(output_file, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([post_url, comments_list, sentiment_labels])  # 写入一行数据

        # 定期输出进度
        if index % 10 == 0:
            print(f"Checkpoint at index {index}, progress saved.")

In [28]:
input_file = 'phd data/merged post comments.csv'
output_file = 'results/comments sentiment.csv'

In [29]:
analyze_comments(input_file, output_file, start_index=286)

Processing index 286
Processing index 287
Processing index 288
Processing index 289
Processing index 290
Checkpoint at index 290, progress saved.
Processing index 291
Processing index 292
Processing index 293
Processing index 294
Processing index 295
Processing index 296
Processing index 297
Processing index 298
Processing index 299
Processing index 300
Checkpoint at index 300, progress saved.
Processing index 301
Processing index 302
Processing index 303
Processing index 304
Processing index 305
Processing index 306
Processing index 307
Processing index 308
Processing index 309
Processing index 310
Checkpoint at index 310, progress saved.
Processing index 311
Processing index 312
Processing index 313
Processing index 314
Processing index 315
Processing index 316
Processing index 317
Error processing text: The expanded size of the tensor (590) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 590].  Tensor sizes: [1, 514]
Processing index 318
Processin

Processing index 530
Checkpoint at index 530, progress saved.
Processing index 531
Processing index 532
Processing index 533
Processing index 534
Processing index 535
Processing index 536
Processing index 537
Processing index 538
Processing index 539
Processing index 540
Checkpoint at index 540, progress saved.
Processing index 541
Processing index 542
Processing index 543
Processing index 544
Processing index 545
Processing index 546
Processing index 547
Processing index 548
Processing index 549
Processing index 550
Checkpoint at index 550, progress saved.
Processing index 551
Processing index 552
Processing index 553
Processing index 554
Processing index 555
Processing index 556
Processing index 557
Processing index 558
Processing index 559
Processing index 560
Checkpoint at index 560, progress saved.
Processing index 561
Processing index 562
Processing index 563
Processing index 564
Processing index 565
Processing index 566
Processing index 567
Processing index 568
Processing index 

Checkpoint at index 820, progress saved.
Processing index 821
Processing index 822
Processing index 823
Processing index 824
Processing index 825
Processing index 826
Processing index 827
Processing index 828
Processing index 829
Processing index 830
Checkpoint at index 830, progress saved.
Processing index 831
Processing index 832
Processing index 833
Processing index 834
Processing index 835
Processing index 836
Processing index 837
Processing index 838
Processing index 839
Processing index 840
Checkpoint at index 840, progress saved.
Processing index 841
Processing index 842
Processing index 843
Processing index 844
Processing index 845
Processing index 846
Processing index 847
Processing index 848
Processing index 849
Processing index 850
Checkpoint at index 850, progress saved.
Processing index 851
Processing index 852
Processing index 853
Processing index 854
Processing index 855
Processing index 856
Processing index 857
Processing index 858
Processing index 859
Processing index 

Error processing text: The expanded size of the tensor (523) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 523].  Tensor sizes: [1, 514]
Error processing text: The expanded size of the tensor (714) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 714].  Tensor sizes: [1, 514]
Processing index 1076
Processing index 1077
Processing index 1078
Processing index 1079
Processing index 1080
Error processing text: The expanded size of the tensor (545) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 545].  Tensor sizes: [1, 514]
Checkpoint at index 1080, progress saved.
Processing index 1081
Processing index 1082
Processing index 1083
Processing index 1084
Processing index 1085
Processing index 1086
Processing index 1087
Processing index 1088
Processing index 1089
Processing index 1090
Checkpoint at index 1090, progress saved.
Processing index 1091
Processing index 1092
Processing index 

Processing index 1336
Processing index 1337
Processing index 1338
Processing index 1339
Processing index 1340
Checkpoint at index 1340, progress saved.
Processing index 1341
Processing index 1342
Processing index 1343
Processing index 1344
Processing index 1345
Processing index 1346
Processing index 1347
Processing index 1348
Processing index 1349
Processing index 1350
Error processing text: The expanded size of the tensor (584) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 584].  Tensor sizes: [1, 514]
Checkpoint at index 1350, progress saved.
Processing index 1351
Processing index 1352
Processing index 1353
Processing index 1354
Processing index 1355
Processing index 1356
Processing index 1357
Processing index 1358
Processing index 1359
Processing index 1360
Checkpoint at index 1360, progress saved.
Processing index 1361
Processing index 1362
Processing index 1363
Processing index 1364
Processing index 1365
Processing index 1366
Processing index 

...

Processing index 6862
Processing index 6863
Processing index 6864
Processing index 6865
Processing index 6866
Processing index 6867
Processing index 6868
Processing index 6869
Processing index 6870
Checkpoint at index 6870, progress saved.
Processing index 6871
Processing index 6872
Processing index 6873
Processing index 6874
Processing index 6875
Processing index 6876
Processing index 6877
Processing index 6878
Processing index 6879
Processing index 6880
Checkpoint at index 6880, progress saved.
Processing index 6881
Processing index 6882
Processing index 6883
Processing index 6884
Processing index 6885
Processing index 6886
Processing index 6887
Processing index 6888
Processing index 6889
Processing index 6890
Checkpoint at index 6890, progress saved.
Processing index 6891
Processing index 6892
Processing index 6893
Processing index 6894
Processing index 6895
Error processing text: The expanded size of the tensor (525) must match the existing size (514) at non-singleton dimension 1. 

Processing index 7102
Processing index 7103
Processing index 7104
Processing index 7105
Processing index 7106
Processing index 7107
Processing index 7108
Processing index 7109
Processing index 7110
Checkpoint at index 7110, progress saved.
Processing index 7111
Processing index 7112
Processing index 7113
Processing index 7114
Processing index 7115
Processing index 7116
Processing index 7117
Processing index 7118
Processing index 7119
Processing index 7120
Checkpoint at index 7120, progress saved.
Processing index 7121
Processing index 7122
Processing index 7123
Processing index 7124
Processing index 7125
Processing index 7126
Processing index 7127
Processing index 7128
Processing index 7129
Processing index 7130
Checkpoint at index 7130, progress saved.
Processing index 7131
Processing index 7132
Processing index 7133
Processing index 7134
Processing index 7135
Processing index 7136
Processing index 7137
Processing index 7138
Processing index 7139
Processing index 7140
Checkpoint at in

Processing index 7349
Processing index 7350
Checkpoint at index 7350, progress saved.
Processing index 7351
Processing index 7352
Processing index 7353
Processing index 7354
Processing index 7355
Processing index 7356
Processing index 7357
Processing index 7358
Processing index 7359
Processing index 7360
Checkpoint at index 7360, progress saved.
Processing index 7361
Processing index 7362
Processing index 7363
Processing index 7364
Processing index 7365
Processing index 7366
Error processing text: The expanded size of the tensor (742) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 742].  Tensor sizes: [1, 514]
Processing index 7367
Processing index 7368
Processing index 7369
Processing index 7370
Error processing text: The expanded size of the tensor (574) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 574].  Tensor sizes: [1, 514]
Checkpoint at index 7370, progress saved.
Processing index 7371
Processing index 7

Processing index 7586
Processing index 7587
Processing index 7588
Processing index 7589
Processing index 7590
Error processing text: The expanded size of the tensor (542) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 542].  Tensor sizes: [1, 514]
Checkpoint at index 7590, progress saved.
Processing index 7591
Error processing text: The expanded size of the tensor (847) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 847].  Tensor sizes: [1, 514]
Error processing text: The expanded size of the tensor (592) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 592].  Tensor sizes: [1, 514]
Processing index 7592
Processing index 7593
Processing index 7594
Processing index 7595
Processing index 7596
Processing index 7597
Processing index 7598
Processing index 7599
Processing index 7600
Checkpoint at index 7600, progress saved.
Processing index 7601
Error processing text: The expanded siz

Processing index 7785
Processing index 7786
Processing index 7787
Processing index 7788
Processing index 7789
Processing index 7790
Checkpoint at index 7790, progress saved.
Processing index 7791
Processing index 7792
Processing index 7793
Processing index 7794
Processing index 7795
Processing index 7796
Processing index 7797
Processing index 7798
Processing index 7799
Processing index 7800
Checkpoint at index 7800, progress saved.
Processing index 7801
Processing index 7802
Processing index 7803
Processing index 7804
Processing index 7805
Processing index 7806
Processing index 7807
Processing index 7808
Processing index 7809
Processing index 7810
Checkpoint at index 7810, progress saved.
Processing index 7811
Processing index 7812
Processing index 7813
Processing index 7814
Processing index 7815
Processing index 7816
Processing index 7817
Processing index 7818
Processing index 7819
Processing index 7820
Error processing text: The expanded size of the tensor (570) must match the existi

In [30]:
df=pd.read_csv(output_file)
df.head()

,post url,comments_list,sentiment_labels
0,Ca79j-DruOL,"['Europe is preparing to celebrate Easter, Ukr...","['neutral', 'neutral', 'negative', 'neutral', ..."
1,Ca2VIVgLeTJ,"['👏👏👏👏👏', 'In einer Zeit, in der viele Länder ...","['positive', 'neutral', 'negative', 'negative'..."
2,CacSefPN61Z,['We are Ukrainians and we need your help! rus...,"['negative', 'neutral', 'negative', 'negative'..."
3,CaXprHNr5zA,"['In einer Zeit, in der viele Länder russische...","['neutral', 'negative', 'positive', 'negative'..."
4,CaFSWMIlyEW,"['👏👏👏', '👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻', 'Unglaublich.... das i...","['positive', 'positive', 'neutral', 'neutral',..."


In [32]:
df.shape

(7998, 3)

In [35]:
df1=pd.read_csv('results/comments_classification_results.csv', encoding='utf-8')
df1.shape
df1.head()

,post url,comment category
0,CacSefPN61Z,['comment does not belong to above categories'...
1,CaXprHNr5zA,"['comments about policy', 'comments about poli..."
2,CaFSWMIlyEW,['comment does not belong to above categories'...
3,CZ7MMC9tcSG,['comment does not belong to above categories'...
4,CZ30zPmtvT6,"['comments about policy', 'comments about poli..."


In [36]:
df=pd.merge(df, df1, on='post url', how='outer')
df.head()

,post url,comments_list,sentiment_labels,comment category
0,CM-I1L-hEPb,"['FDP-Eis 🤩💪🏻', '💗', 'Bei @katringrothe ist ab...","['positive', 'positive', 'neutral', 'positive'...",['comments about political party or organizati...
1,CM-KytsIXhb,"['😍', 'Endlich Mal neue Ideen umsetzen und sic...","['positive', 'neutral', 'positive', 'positive'...","['comments about event', 'comments about polic..."
2,CM-XYoBoqpR,['Herr Ziemiak . Ihre Ablehnung eines harten L...,"['neutral', None, 'neutral', 'neutral', 'neutr...","['comments about person or politician', 'comme..."
3,CM-fOWwhGHO,"['Hoffentlich regieren sie nicht, das wird son...","['neutral', 'positive', 'neutral', 'positive',...",['comments about political party or organizati...
4,CM3VunOh6bV,[],[],[]


In [38]:
df.rename(columns={'comment category': 'comment_category'}, inplace=True)
df.head()

,post url,comments_list,comment_sentiment,comment_category
0,CM-I1L-hEPb,"['FDP-Eis 🤩💪🏻', '💗', 'Bei @katringrothe ist ab...","['positive', 'positive', 'neutral', 'positive'...",['comments about political party or organizati...
1,CM-KytsIXhb,"['😍', 'Endlich Mal neue Ideen umsetzen und sic...","['positive', 'neutral', 'positive', 'positive'...","['comments about event', 'comments about polic..."
2,CM-XYoBoqpR,['Herr Ziemiak . Ihre Ablehnung eines harten L...,"['neutral', None, 'neutral', 'neutral', 'neutr...","['comments about person or politician', 'comme..."
3,CM-fOWwhGHO,"['Hoffentlich regieren sie nicht, das wird son...","['neutral', 'positive', 'neutral', 'positive',...",['comments about political party or organizati...
4,CM3VunOh6bV,[],[],[]


In [39]:
df.to_csv('results/comments results.csv', encoding='utf-8')

In [60]:
df=pd.read_csv('results/comments results.csv', encoding='utf-8')

In [61]:
print(df['comment_sentiment'][1:2])

1    ['positive', 'neutral', 'positive', 'positive'...
Name: comment_sentiment, dtype: object


In [48]:
# 将每个列表转换为一个字典，键为情绪，值为该情绪出现的次数
df['emotion_dict'] = df['comment_sentiment'].apply(lambda emotions: {emotion: emotions.count(emotion) for emotion in emotions})

# 使用pd.Series将字典列展开成多个列，然后使用fillna填充NaN值为0
emotion_df = df['emotion_dict'].apply(pd.Series).fillna(0)

# 将生成的情绪频率列合并回原始DataFrame
df_expanded = pd.concat([df, emotion_df], axis=1)

# 删除中间生成的emotion_dict列
df_expanded.drop('emotion_dict', axis=1, inplace=True)

In [68]:
from collections import Counter
from ast import literal_eval

# 如果 'comment_sentiment' 是字符串格式的列表，先将其转换为实际列表
df['comment_sentiment'] = df['comment_sentiment'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

# 使用 Counter 来统计每个情绪标签的频率
df['emotion_dict'] = df['comment_sentiment'].apply(lambda emotions: Counter(emotions))

df['emotion_dict'].head()

0              {'positive': 4, 'neutral': 3}
1    {'positive': 10, 'neutral': 4, None: 1}
2                    {'neutral': 8, None: 2}
3              {'neutral': 8, 'positive': 7}
4                                         {}
Name: emotion_dict, dtype: object

In [69]:
# 2. 使用 pd.json_normalize 将 'emotion_dict' 列展开成单独的情绪频率列
emotion_df = pd.json_normalize(df['emotion_dict']).fillna(0).astype(int)

# 3. 将生成的情绪频率列合并回原始 DataFrame
df_expanded = pd.concat([df, emotion_df], axis=1)

# 4. 删除中间生成的 'emotion_dict' 列
df_expanded.drop('emotion_dict', axis=1, inplace=True)

In [72]:
# 展示结果
df_expanded[:5]

,Unnamed: 0,post url,comments_list,comment_sentiment,comment_category,positive,neutral,None,negative
0,0,CM-I1L-hEPb,"['FDP-Eis 🤩💪🏻', '💗', 'Bei @katringrothe ist ab...","[positive, positive, neutral, positive, neutra...",['comments about political party or organizati...,4,3,0,0
1,1,CM-KytsIXhb,"['😍', 'Endlich Mal neue Ideen umsetzen und sic...","[positive, neutral, positive, positive, positi...","['comments about event', 'comments about polic...",10,4,1,0
2,2,CM-XYoBoqpR,['Herr Ziemiak . Ihre Ablehnung eines harten L...,"[neutral, None, neutral, neutral, neutral, neu...","['comments about person or politician', 'comme...",0,8,2,0
3,3,CM-fOWwhGHO,"['Hoffentlich regieren sie nicht, das wird son...","[neutral, positive, neutral, positive, neutral...",['comments about political party or organizati...,7,8,0,0
4,4,CM3VunOh6bV,[],[],[],0,0,0,0


In [75]:
df = df.merge(df_expanded[['post url', 'positive', 'neutral', 'negative']], on='post url', how='left')

In [76]:
df.head()

,Unnamed: 0,post url,comments_list,comment_sentiment,comment_category,emotion_dict,positive,neutral,negative
0,0,CM-I1L-hEPb,"['FDP-Eis 🤩💪🏻', '💗', 'Bei @katringrothe ist ab...","[positive, positive, neutral, positive, neutra...",['comments about political party or organizati...,"{'positive': 4, 'neutral': 3}",4,3,0
1,1,CM-KytsIXhb,"['😍', 'Endlich Mal neue Ideen umsetzen und sic...","[positive, neutral, positive, positive, positi...","['comments about event', 'comments about polic...","{'positive': 10, 'neutral': 4, None: 1}",10,4,0
2,2,CM-XYoBoqpR,['Herr Ziemiak . Ihre Ablehnung eines harten L...,"[neutral, None, neutral, neutral, neutral, neu...","['comments about person or politician', 'comme...","{'neutral': 8, None: 2}",0,8,0
3,3,CM-fOWwhGHO,"['Hoffentlich regieren sie nicht, das wird son...","[neutral, positive, neutral, positive, neutral...",['comments about political party or organizati...,"{'neutral': 8, 'positive': 7}",7,8,0
4,4,CM3VunOh6bV,[],[],[],{},0,0,0


In [79]:
# 如果 'comment_category' 是字符串格式的列表，先将其转换为实际列表，并处理 NaN 值
df['comment_category'] = df['comment_category'].apply(lambda x: literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else []))

# 使用 Counter 来统计每个标签的频率
df['category_dict'] = df['comment_category'].apply(lambda emotions: Counter(emotions))

# 使用 pd.json_normalize 将 'category_dict' 列展开成单独的频率列
category_df = pd.json_normalize(df['category_dict']).fillna(0).astype(int)

# 将生成的频率列合并回原始 DataFrame
df_expanded1 = pd.concat([df, category_df], axis=1)

# 删除中间生成的 'category_dict' 列
df_expanded1.drop('category_dict', axis=1, inplace=True)
df_expanded1.head()

,Unnamed: 0,post url,comments_list,comment_sentiment,comment_category,emotion_dict,positive,neutral,negative,comments about political party or organization,comment does not belong to above categories,comments about event,comments about policy,comments about person or politician
0,0,CM-I1L-hEPb,"['FDP-Eis 🤩💪🏻', '💗', 'Bei @katringrothe ist ab...","[positive, positive, neutral, positive, neutra...",[comments about political party or organizatio...,"{'positive': 4, 'neutral': 3}",4,3,0,1,6,0,0,0
1,1,CM-KytsIXhb,"['😍', 'Endlich Mal neue Ideen umsetzen und sic...","[positive, neutral, positive, positive, positi...","[comments about event, comments about policy, ...","{'positive': 10, 'neutral': 4, None: 1}",10,4,0,3,0,7,3,2
2,2,CM-XYoBoqpR,['Herr Ziemiak . Ihre Ablehnung eines harten L...,"[neutral, None, neutral, neutral, neutral, neu...","[comments about person or politician, comments...","{'neutral': 8, None: 2}",0,8,0,4,0,1,4,1
3,3,CM-fOWwhGHO,"['Hoffentlich regieren sie nicht, das wird son...","[neutral, positive, neutral, positive, neutral...",[comments about political party or organizatio...,"{'neutral': 8, 'positive': 7}",7,8,0,7,0,4,1,3
4,4,CM3VunOh6bV,[],[],[],{},0,0,0,0,0,0,0,0


In [85]:
df_expanded1.shape

(8239, 14)

In [86]:
df_expanded1.to_csv('comments analysis.csv', encoding='utf-8')